# 🗺️ ICE Raid Buffer Analysis — Combined Overview
**Run cells top to bottom.**
- **Cell 1** — Imports, config, spatial helpers, chart functions  
- **Cell 2** — Enhanced stats functions (includes `generate_overview_table` with `Raid_ID` column)  
- **Cell 3** — Execution functions + interactive widget  

After running a selection the combined overview table is displayed inline and saved to `{OUT_DIR}/combined_overview_selected.csv`.

In [29]:

import os
import glob
import gzip
import json
import time
import pickle
from datetime import timedelta
import warnings

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.font_manager as fm
from matplotlib.patches import Patch
from IPython.display import display, clear_output, Image as IPyImage

import ipywidgets as widgets


# ============================================
# CONFIG
# ============================================

# Base path where all inputs and foot-traffic data live
INPUT_BASE = os.environ.get('ICE_INPUT_BASE', '/primary/work/ice')

# Large foot-traffic CSVs
TEMP_BASE          = os.path.join(INPUT_BASE, 'temp')
WEEKLY_DATA_FOLDER = os.path.join(TEMP_BASE, 'california-foot-traffic-weekly-patterns-plus/california-weekly-data-all/')

LATINO_TRACTS_GEOJSON = os.path.join(INPUT_BASE, 'inputs', 'LACHD_majority_latino_neighborhoods_census_tracts_2020.geojson')
ICE_ACTIVITY_CSV      = os.path.join(INPUT_BASE, 'inputs', 'ICESummerActivityLog2025-geocoded.csv')

# buffer polygons replace census tracts for raid locations
HALF_MILE_GEOJSON    = os.path.join(INPUT_BASE, 'inputs', 'ice-june-6-2025-one-week-half_mile_buffers.geojson')
QUARTER_MILE_GEOJSON = os.path.join(INPUT_BASE, 'inputs', 'ice-june-6-2025-one-week-quarter_mile_buffers.geojson')

# outputs go under INPUT_BASE (same mount, writable)
OUT_DIR   = os.path.join(INPUT_BASE, 'outputs', 'raid_analysis')
CACHE_DIR = os.path.join(OUT_DIR, 'cache')
STATS_DIR = os.path.join(OUT_DIR, 'descriptive_stats')
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(STATS_DIR, exist_ok=True)

# Key Events
KEY_EVENTS = {
    "3/4/2020\nState of\nEmergency": "2020-03-04",
    "3/19/2020\nStatewide\nStay at Home": "2020-03-19",
    "1/25/2022\nEnd of State of Emergency": "2022-01-25",
    "1/07/2025\nLA Fires": "2025-01-07",
    "1/20/2025\nInauguration": "2025-01-20",
    "6/05/2025\nLA ICE Raids": "2025-06-05",
    "9/08/2025\nSCOTUS": "2025-09-08",
    "10/14/2025\nLA County\nImmigration Emergency": "2025-10-14"
}
KEY_EVENTS_2025 = {k: v for k, v in KEY_EVENTS.items() if v >= '2025-01-01'}

NAICS_HIGH_IMPACT = ['1111', '1112', '4451', '7225', '44', '45']
NAICS_HOSPITALS   = ['622110', '6221']

# Human-readable labels for NAICS sectors (matched longest-prefix-first)
NAICS_LABEL_MAP = {
    # Agriculture subsectors (4-digit)
    '1111': 'Oilseed & Grain Farming',
    '1112': 'Vegetable & Melon Farming',
    '1113': 'Fruit & Tree Nut Farming',
    '1114': 'Greenhouse & Nursery',
    '1115': 'Other Crop Farming',
    '1121': 'Cattle Ranching',
    '1122': 'Hog & Pig Farming',
    '1123': 'Poultry & Egg Production',
    # Retail (4-digit)
    '4411': 'Automobile Dealers',
    '4413': 'Auto Parts & Accessories',
    '4421': 'Furniture Stores',
    '4431': 'Electronics & Appliance',
    '4441': 'Building Material & Garden',
    '4451': 'Grocery Stores',
    '4452': 'Specialty Food Stores',
    '4453': 'Beer, Wine & Liquor',
    '4461': 'Health & Personal Care Stores',
    '4471': 'Gasoline Stations',
    '4481': 'Clothing Stores',
    '4482': 'Shoe Stores',
    '4483': 'Jewelry & Luxury',
    '4511': 'Sporting Goods & Hobby',
    '4512': 'Book & Music Stores',
    '4521': 'Department Stores',
    '4529': 'General Merchandise',
    '4531': 'Florists',
    '4532': 'Office Supplies & Gifts',
    '4533': 'Used Merchandise',
    '4539': 'Other Miscellaneous Retail',
    # Food services (4-digit)
    '7221': 'Full-Service Restaurants',
    '7222': 'Limited-Service Eating Places',
    '7223': 'Special Food Services',
    '7224': 'Drinking Places (Bars)',
    '7225': 'Restaurants & Eating Places',
    # Healthcare (4-digit)
    '6211': 'Offices of Physicians',
    '6212': 'Offices of Dentists',
    '6213': 'Other Health Practitioners',
    '6214': 'Outpatient Care Centers',
    '6215': 'Medical & Diagnostic Labs',
    '6216': 'Home Health Care Services',
    '6219': 'Other Ambulatory Health',
    '6221': 'General Medical Hospitals',
    '6222': 'Psychiatric Hospitals',
    '6223': 'Specialty Hospitals',
    '6231': 'Nursing Care Facilities',
    '6232': 'Residential Care Facilities',
    # Education (4-digit)
    '6111': 'Elementary & Secondary Schools',
    '6112': 'Junior Colleges',
    '6113': 'Colleges & Universities',
    '6114': 'Business Schools',
    '6115': 'Technical & Trade Schools',
    '6116': 'Other Schools & Instruction',
    # Personal & Other Services (4-digit)
    '8121': 'Personal Care Services',
    '8122': 'Death Care Services',
    '8123': 'Drycleaning & Laundry',
    '8129': 'Other Personal Services',
    '8131': 'Religious Organizations',
    '8132': 'Grantmaking & Giving',
    '8134': 'Civic & Social Organizations',
    # Sector-level fallbacks (2-digit)
    '11': 'Agriculture & Forestry',
    '21': 'Mining',
    '22': 'Utilities',
    '23': 'Construction',
    '31': 'Manufacturing',
    '32': 'Manufacturing',
    '33': 'Manufacturing',
    '42': 'Wholesale Trade',
    '44': 'Retail Trade',
    '45': 'Retail Trade (Misc.)',
    '48': 'Transportation',
    '49': 'Warehousing',
    '51': 'Information',
    '52': 'Finance & Insurance',
    '53': 'Real Estate',
    '54': 'Professional & Tech Services',
    '55': 'Management',
    '56': 'Admin & Support',
    '61': 'Education',
    '62': 'Health & Social Services',
    '71': 'Arts & Entertainment',
    '72': 'Accommodation & Food',
    '81': 'Other Services',
    '92': 'Public Administration',
}


# ============================================
# FONT SETUP
# ============================================

lexend_path = os.path.join(INPUT_BASE, 'fonts', 'Lexend-Regular.ttf')
if os.path.exists(lexend_path):
    fm.fontManager.addfont(lexend_path)
    plt.rcParams["font.family"] = "Lexend"

plt.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 300, 'axes.titlesize': 16,
    'axes.labelsize': 14, 'xtick.labelsize': 11, 'ytick.labelsize': 11
})


def log_step(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")


# ============================================
# INPUT VALIDATION
# ============================================

paths_to_check = {
    "WEEKLY_DATA_FOLDER": WEEKLY_DATA_FOLDER,
    "LATINO_TRACTS_GEOJSON": LATINO_TRACTS_GEOJSON,
    "HALF_MILE_GEOJSON": HALF_MILE_GEOJSON,
    "QUARTER_MILE_GEOJSON": QUARTER_MILE_GEOJSON,
}
for label, path in paths_to_check.items():
    status = "✅" if os.path.exists(path) else "❌ MISSING"
    print(f"{status}  {label}: {path}")
print(f"✅  OUT_DIR: {OUT_DIR}")


# ============================================
# 🏷️ EVENT LABELS
# ============================================

def add_readable_event_labels(ax, events=KEY_EVENTS):
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin, ymax * 1.15)
    base_color = 'gray'
    arrowprops = dict(arrowstyle='-', color=base_color, lw=0.8, alpha=0.7)
    sorted_items = sorted(events.items(), key=lambda kv: pd.to_datetime(kv[1]))
    placed = []
    for i, (label, date_str) in enumerate(sorted_items):
        x = pd.to_datetime(date_str)
        alpha = 0.4
        if placed:
            prev_x, _ = placed[-1]
            delta = (x - prev_x).days
            if delta < 30:
                alpha = 0.9
            elif delta < 90:
                alpha = 0.7
        ax.axvline(x, color=base_color, ls='--', lw=2.5, alpha=alpha)
        y_off = 10 + (i * 2)
        for px, py in placed:
            if abs((x - px).days) < 30:
                y_off += 8
        placed.append((x, y_off))
        x_offset = (-35 if i % 2 == 0 else 35)
        ax.annotate(label, xy=(x, ymax), xytext=(x_offset, y_off - 5),
                   textcoords='offset points', ha='center', va='bottom', rotation=90,
                   fontsize=7, fontweight='bold', color=base_color,
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, ec=base_color),
                   annotation_clip=False, arrowprops=arrowprops)


# ============================================
# 🎯 LOAD BUFFER POLYGONS
# ============================================

def get_buffer_polygons():
    half    = gpd.read_file(HALF_MILE_GEOJSON).to_crs('EPSG:4326')
    quarter = gpd.read_file(QUARTER_MILE_GEOJSON).to_crs('EPSG:4326')
    if 'ID' in half.columns:
        half = half.drop_duplicates(subset=['ID'], keep='first').reset_index(drop=True)
    log_step(f"🔍 Loaded {len(half)} half-mile buffers and {len(quarter)} quarter-mile buffers")
    return half, quarter


# ============================================
# 🔍 SPATIAL AGGREGATION
# ============================================

def parse_visits_fast(visits_str):
    if pd.isna(visits_str) or not isinstance(visits_str, str):
        return 0
    try:
        return sum(json.loads(visits_str))
    except:
        return 0


def get_naics_codes(high_impact_include_ag):
    if high_impact_include_ag:
        return [NAICS_HIGH_IMPACT]
    else:
        return [['4451', '7225', '44', '45']]


def _read_csv_robust(fp, cols, chunksize=50000):
    engines = ['c', 'python']
    for eng in engines:
        try:
            read_kw = dict(chunksize=chunksize, usecols=cols, on_bad_lines='skip',
                           engine=eng, dtype={c: str for c in cols})
            if eng == 'c':
                read_kw['low_memory'] = False
            for chunk_df in pd.read_csv(fp, **read_kw):
                for coord in ('LONGITUDE', 'LATITUDE'):
                    if coord in chunk_df.columns:
                        chunk_df[coord] = pd.to_numeric(chunk_df[coord], errors='coerce')
                yield chunk_df
            return
        except (OSError, gzip.BadGzipFile, EOFError) as e:
            log_step(f"⚠️ Corrupt gzip {os.path.basename(fp)}: {str(e)[:80]}")
            return
        except Exception as e:
            err_msg = str(e)
            if 'crc' in err_msg.lower() or 'decompression' in err_msg.lower():
                log_step(f"⚠️ Corrupt gzip {os.path.basename(fp)}: {err_msg[:80]}")
                return
            if eng == 'c' and ('buffer overflow' in err_msg.lower()
                               or 'Error tokenizing' in err_msg
                               or 'could not convert' in err_msg.lower()
                               or "expected after" in err_msg.lower()):
                log_step(f"⚠️ C-engine failed on {os.path.basename(fp)}, retrying with Python engine")
                continue
            if eng == 'python':
                log_step(f"⚠️ Skipping {os.path.basename(fp)}: {err_msg[:100]}")
                return
            continue


def spatial_aggregate(weekly_files, area_gdf, area_label, naics_filter=None,
                      cache_file=None, collect_naics=False, naics_cache_file=None):
    """Aggregate weekly foot traffic for an area.

    Returns (DataFrame, n_pois, naics_df) where naics_df is a per-sector breakdown
    collected for free during the same pass when collect_naics=True (no extra I/O).
    naics_df is None when collect_naics=False or when loaded from a legacy cache.

    Cache files store {'df': ..., 'n_pois': ...}; NAICS breakdown is cached separately
    via naics_cache_file. Tick 'Clear cache' to regenerate legacy caches with n= counts.
    """
    naics_df = None

    if cache_file and os.path.exists(cache_file):
        try:
            with open(cache_file, 'rb') as _f:
                _cached = pickle.load(_f)
            # Try loading NAICS sidecar cache
            if collect_naics and naics_cache_file and os.path.exists(naics_cache_file):
                try:
                    naics_df = pd.read_pickle(naics_cache_file)
                except Exception:
                    pass
            if isinstance(_cached, dict):
                _df = _cached['df']
                if 'n_pois' not in _df.columns:
                    _df = _df.assign(n_pois=None)   # legacy cache — tick 'Clear cache' to regenerate
                log_step(f"✅ Cache: {area_label}  (n_pois={_cached.get('n_pois'):,})")
                return _df, _cached.get('n_pois'), naics_df
            # Legacy cache: no n_pois stored
            log_step(f"✅ Cache (legacy — tick 'Clear cache' to add n= counts): {area_label}")
            return _cached, None, naics_df
        except Exception:
            pass

    if area_gdf.empty:
        log_step(f"❌ EMPTY AREA GDF: {area_label}")
        return pd.DataFrame(), 0, None

    join_cols = ['geometry']
    if 'GEOID' in area_gdf.columns:
        join_cols = ['GEOID', 'geometry']
    join_gdf = area_gdf[join_cols].copy()
    log_step(f"🔄 {area_label} ({len(area_gdf)} polygons, {len(weekly_files)} files)...")
    records          = []
    poi_set          = set()
    weekly_poi_sets  = {}          # week_start -> set of (lat_r4, lon_r4)
    naics_data = {} if collect_naics else None   # collected in-pass at no extra I/O cost
    total_files = len(weekly_files)
    for fi, fp in enumerate(weekly_files):
        if fi % 30 == 0:
            log_step(f"  {fi}/{total_files} ({fi/total_files*100:.0f}%)")
        try:
            cols = ['DATE_RANGE_START', 'VISITS_BY_DAY', 'LONGITUDE', 'LATITUDE']
            if naics_filter or collect_naics:
                cols.append('NAICS_CODE')
            for chunk_df in _read_csv_robust(fp, cols):
                if chunk_df.empty:
                    continue
                chunk_df = chunk_df.dropna(subset=['LONGITUDE', 'LATITUDE'])
                if chunk_df.empty:
                    continue
                chunk_df = chunk_df[
                    (chunk_df['LONGITUDE'].between(-180, 180)) &
                    (chunk_df['LATITUDE'].between(-90, 90))
                ]
                if chunk_df.empty:
                    continue
                gdf = gpd.GeoDataFrame(
                    chunk_df,
                    geometry=gpd.points_from_xy(chunk_df['LONGITUDE'], chunk_df['LATITUDE']),
                    crs='EPSG:4326'
                )
                gdf = gdf[gdf.geometry.is_valid]
                if gdf.empty:
                    continue
                try:
                    joined = gpd.sjoin(gdf, join_gdf, how='inner', predicate='within')
                except Exception as e:
                    log_step(f"⚠️ Spatial join failed: {str(e)[:100]}")
                    continue
                if joined.empty:
                    continue
                if naics_filter and 'NAICS_CODE' in joined.columns:
                    codes = [c for sector in naics_filter for c in sector]
                    naics_mask = joined['NAICS_CODE'].astype(str).str[:4].isin(codes)
                    joined = joined[naics_mask]
                if joined.empty:
                    continue
                # Track unique POIs by rounded lat/lon
                poi_set.update(zip(
                    joined['LATITUDE'].round(4),
                    joined['LONGITUDE'].round(4)
                ))
                joined['DATE_RANGE_START'] = pd.to_datetime(joined['DATE_RANGE_START'], errors='coerce')
                joined = joined.dropna(subset=['DATE_RANGE_START'])
                joined['week_start'] = joined['DATE_RANGE_START'].dt.normalize()
                joined['visits'] = joined['VISITS_BY_DAY'].apply(parse_visits_fast)
                # Collect NAICS breakdown in the same pass — no extra file reads
                if collect_naics and naics_data is not None and 'NAICS_CODE' in joined.columns:
                    joined['_naics4'] = joined['NAICS_CODE'].astype(str).str[:4]
                    for nc4, grp in joined.groupby('_naics4'):
                        if nc4 not in naics_data:
                            naics_data[nc4] = {'visits': 0, 'pois': set()}
                        naics_data[nc4]['visits'] += int(grp['visits'].sum())
                        naics_data[nc4]['pois'].update(zip(
                            grp['LATITUDE'].round(4), grp['LONGITUDE'].round(4)
                        ))
                # Per-week unique POI count (for n= column on event-study tables)
                for week_ts, wgrp in joined.groupby('week_start'):
                    pts = set(zip(wgrp['LATITUDE'].round(4), wgrp['LONGITUDE'].round(4)))
                    if week_ts not in weekly_poi_sets:
                        weekly_poi_sets[week_ts] = set()
                    weekly_poi_sets[week_ts].update(pts)
                weekly = joined.groupby('week_start')['visits'].sum()
                records.extend([{'week_start': w, 'total_visits': v} for w, v in weekly.items() if v > 0])
        except Exception as e:
            log_step(f"⚠️ File {fi} error: {str(e)[:80]}")
            continue
    df = pd.DataFrame(records)
    if df.empty:
        log_step(f"⚠️ NO DATA for {area_label}")
        return pd.DataFrame(), 0, None
    result = df.groupby('week_start')['total_visits'].sum().reset_index().sort_values('week_start')
    n_pois = len(poi_set)
    # Attach per-week unique POI counts as a column so event-study tables can show n=
    n_per_week = {w: len(s) for w, s in weekly_poi_sets.items()}
    result['n_pois'] = result['week_start'].map(n_per_week).fillna(0).astype(int)
    # Build and cache NAICS breakdown if collected
    if collect_naics and naics_data:
        naics_rows = [
            {'naics4': k, 'total_visits': v['visits'], 'n_pois': len(v['pois'])}
            for k, v in naics_data.items()
        ]
        naics_df = pd.DataFrame(naics_rows).sort_values('total_visits', ascending=False).reset_index(drop=True)
        if naics_cache_file:
            naics_df.to_pickle(naics_cache_file)
        log_step(f"📊 NAICS breakdown: {len(naics_df)} sectors collected in-pass")
    if cache_file:
        with open(cache_file, 'wb') as _f:
            pickle.dump({'df': result, 'n_pois': n_pois}, _f)
    log_step(f"✅ {area_label}: {len(result)} weeks, {n_pois:,} unique POIs")
    return result, n_pois, naics_df


# ============================================
# 📊 DESCRIPTIVE STATS (legacy)
# ============================================

def generate_stats(la_df, tract_df, suffix, location_name='Raid Location'):
    if la_df.empty or tract_df.empty:
        return
    stats = []
    stats.append({'Metric': 'Date Range',
                  'LA_County': f"{la_df['week_start'].min().date()} to {la_df['week_start'].max().date()}",
                  location_name: f"{tract_df['week_start'].min().date()} to {tract_df['week_start'].max().date()}"})
    stats.append({'Metric': 'Total Weeks', 'LA_County': len(la_df), location_name: len(tract_df)})
    stats.append({'Metric': 'Mean Weekly Visits',
                  'LA_County': f"{la_df['total_visits'].mean():.0f}",
                  location_name: f"{tract_df['total_visits'].mean():.0f}"})
    stats.append({'Metric': 'Median Weekly Visits',
                  'LA_County': f"{la_df['total_visits'].median():.0f}",
                  location_name: f"{tract_df['total_visits'].median():.0f}"})
    pre_la    = la_df[la_df['week_start'] < '2025-01-01']['total_visits']
    pre_tract = tract_df[tract_df['week_start'] < '2025-01-01']['total_visits']
    stats.append({'Metric': 'Pre-2025 Mean',
                  'LA_County': f"{pre_la.mean():.0f}",
                  location_name: f"{pre_tract.mean():.0f}"})
    post_la    = la_df[la_df['week_start'] >= '2025-06-01']['total_visits']
    post_tract = tract_df[tract_df['week_start'] >= '2025-06-01']['total_visits']
    stats.append({'Metric': 'Post-Raid Mean (Jun+)',
                  'LA_County': f"{post_la.mean():.0f}",
                  location_name: f"{post_tract.mean():.0f}"})
    if len(pre_la) > 0 and len(post_la) > 0:
        la_change    = ((post_la.mean() / pre_la.mean()) - 1) * 100
        tract_change = ((post_tract.mean() / pre_tract.mean()) - 1) * 100
        stats.append({'Metric': 'Post/Pre % Change',
                      'LA_County': f"{la_change:.1f}%",
                      location_name: f"{tract_change:.1f}%"})
    pd.DataFrame(stats).to_csv(os.path.join(STATS_DIR, f'stats_{suffix}.csv'), index=False)
    log_step(f"📊 Stats CSV: stats_{suffix}.csv")


# ============================================
# 🎨 CHART FUNCTIONS
# ============================================

def _configure_time_axis(ax):
    start, end = pd.to_datetime('2019-01-01'), pd.to_datetime('2025-12-31')
    ax.set_xlim(start, end)
    years = pd.date_range(start, end, freq='YS')
    ax.set_xticks(years)
    ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[6]))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_minor_formatter(mdates.DateFormatter(''))
    ax.tick_params(axis='x', which='minor', length=5, color='lightgray')
    ax.grid(which='major', alpha=0.3)
    ax.grid(which='minor', color='lightgray', linestyle=':', alpha=0.5)


def _configure_2025_axis(ax):
    start, end = pd.to_datetime('2025-01-01'), pd.to_datetime('2025-12-31')
    ax.set_xlim(start, end)
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.xaxis.set_minor_locator(mdates.WeekdayLocator())
    ax.tick_params(axis='x', which='minor', length=3, color='lightgray')
    ax.grid(which='major', alpha=0.3)
    ax.grid(which='minor', color='lightgray', linestyle=':', alpha=0.3)


def _n_suffix(n_pois):
    """Return ' (n=X)' string if n_pois is available, else empty string."""
    return f' (n={n_pois:,})' if n_pois is not None else ''


def create_single_chart(df, title, subtitle, color, save_path, n_pois=None):
    if df.empty:
        return
    fig, ax = plt.subplots(figsize=(15, 9))
    ax.plot(df['week_start'], df['total_visits'], linewidth=4, color=color, marker='o', markersize=5)
    subtitle_line = f'{subtitle}{_n_suffix(n_pois)}' if subtitle else _n_suffix(n_pois).strip()
    ax.set_title(f'{title}\n{subtitle_line}', fontsize=20, fontweight='bold')
    ax.set_ylabel('Weekly Visits')
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'{int(x):,}'))
    _configure_time_axis(ax)
    add_readable_event_labels(ax)
    plt.xticks(rotation=45)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()


def create_comparison_chart(la_df, tract_df, fips, location_name, save_path, subtitle='',
                             n_pois_la=None, n_pois_area=None):
    if la_df.empty or tract_df.empty:
        return
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), height_ratios=[3, 1])
    la_base    = la_df[la_df['week_start'] < '2020-03-01']['total_visits'].mean()
    tract_base = tract_df[tract_df['week_start'] < '2020-03-01']['total_visits'].mean()
    if la_base == 0 or tract_base == 0:
        return
    la_norm    = la_df['total_visits'] / la_base * 100
    tract_norm = tract_df['total_visits'] / tract_base * 100
    ax1.plot(la_df['week_start'], la_norm, '#1f77b4', lw=4, marker='o', markersize=5)
    ax1.plot(tract_df['week_start'], tract_norm, '#d62728', lw=4, marker='s', markersize=5)
    sub_with_n = f'{subtitle}{_n_suffix(n_pois_area)}' if subtitle else f'Area{_n_suffix(n_pois_area)}'
    ax1.set_title(f'LA vs {location_name} {sub_with_n}\nIndexed to 2019 (100)',
                  fontweight='bold', fontsize=18)
    ax1.set_ylabel('Indexed Traffic')
    ax1.legend(['LA County', location_name])
    _configure_time_axis(ax1)
    add_readable_event_labels(ax1)
    common_dates = pd.merge(la_df[['week_start']], tract_df[['week_start']], on='week_start', how='inner')['week_start']
    if len(common_dates) > 0:
        common_dates = pd.to_datetime(common_dates)
        la_common    = la_df[la_df['week_start'].isin(common_dates)]['total_visits']
        tract_common = tract_df[tract_df['week_start'].isin(common_dates)]['total_visits']
        diff = tract_common.values - la_common.values
        ax2.fill_between(common_dates, diff, 0, color='#d62728', alpha=0.4)
        ax2.axhline(0, color='k', lw=2)
        ax2.set_ylabel('Difference')
    ax2.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()


def create_2025_comparison_chart(la_df, area_df, location_name, save_path, subtitle='',
                                  n_pois_la=None, n_pois_area=None):
    if la_df.empty or area_df.empty:
        return
    la_2025   = la_df[la_df['week_start'] >= '2025-01-01'].copy()
    area_2025 = area_df[area_df['week_start'] >= '2025-01-01'].copy()
    if la_2025.empty or area_2025.empty:
        return
    title_str = f'2025 Weekly Traffic – LA County vs {location_name}'
    if subtitle:
        title_str += f'\n{subtitle}{_n_suffix(n_pois_area)}'
    elif n_pois_area is not None:
        title_str += _n_suffix(n_pois_area)
    fig, ax1 = plt.subplots(figsize=(16, 8))
    ax2 = ax1.twinx()
    color_la   = '#1f77b4'
    color_area = '#d62728'
    ax1.plot(la_2025['week_start'], la_2025['total_visits'],
             color=color_la, lw=3, marker='o', markersize=5, label='LA County')
    ax2.plot(area_2025['week_start'], area_2025['total_visits'],
             color=color_area, lw=3, marker='s', markersize=5, label=location_name)
    ax1.set_ylabel('LA County Weekly Visits', color=color_la, fontsize=13)
    ax2.set_ylabel(f'{location_name} Weekly Visits', color=color_area, fontsize=13)
    ax1.tick_params(axis='y', labelcolor=color_la)
    ax2.tick_params(axis='y', labelcolor=color_area)
    ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'{int(x):,}'))
    ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'{int(x):,}'))
    _configure_2025_axis(ax1)
    ax1.set_title(title_str, fontsize=18, fontweight='bold')
    ax1.set_xlabel('2025', fontsize=12)
    ymin, ymax = ax1.get_ylim()
    ax1.set_ylim(ymin, ymax * 1.2)
    sorted_events = sorted(KEY_EVENTS_2025.items(), key=lambda kv: pd.to_datetime(kv[1]))
    for i, (label, date_str) in enumerate(sorted_events):
        x = pd.to_datetime(date_str)
        ax1.axvline(x, color='gray', ls='--', lw=1.8, alpha=0.7)
        y_off = 15 + (i % 3) * 20
        ax1.annotate(label, xy=(x, ymax * 1.18), xytext=(0, y_off),
                    textcoords='offset points', ha='center', va='bottom', rotation=90,
                    fontsize=7, fontweight='bold', color='gray',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.85, ec='gray'),
                    annotation_clip=False)
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=11)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    log_step(f"📈 2025 chart saved: {os.path.basename(save_path)}")


def create_naics_pie_chart(naics_df, location_name, save_path, top_n=12):
    """Pie chart of foot-traffic visits broken down by human-readable NAICS sector.

    Data comes free from the spatial_aggregate pass (collect_naics=True) — no extra I/O.
    """
    if naics_df is None or naics_df.empty:
        return

    def _naics_label(code):
        code = str(code)
        for length in (4, 3, 2):
            prefix = code[:length]
            if prefix in NAICS_LABEL_MAP:
                return NAICS_LABEL_MAP[prefix]
        return f'Other (NAICS {code[:2]})'

    df = naics_df.copy()
    df['label'] = df['naics4'].apply(_naics_label)
    by_label = (
        df.groupby('label')
          .agg({'total_visits': 'sum', 'n_pois': 'sum'})
          .sort_values('total_visits', ascending=False)
    )

    # Collapse tail into "Other Sectors"
    if len(by_label) > top_n:
        top          = by_label.head(top_n).copy()
        other_visits = by_label.iloc[top_n:]['total_visits'].sum()
        other_pois   = by_label.iloc[top_n:]['n_pois'].sum()
        if other_visits > 0:
            top.loc['Other Sectors'] = [other_visits, other_pois]
        by_label = top

    total_pois   = int(naics_df['n_pois'].sum())
    total_visits = int(naics_df['total_visits'].sum())

    fig, ax = plt.subplots(figsize=(13, 9))
    cmap   = plt.cm.Set3
    colors = [cmap(i / max(len(by_label) - 1, 1)) for i in range(len(by_label))]
    wedges, _, autotexts = ax.pie(
        by_label['total_visits'],
        labels=None,
        autopct='%1.1f%%',
        colors=colors,
        startangle=90,
        pctdistance=0.75,
    )
    for at in autotexts:
        at.set_fontsize(9)

    legend_labels = [
        f"{lbl} (n={int(row['n_pois']):,})"
        for lbl, row in by_label.iterrows()
    ]
    ax.legend(
        wedges, legend_labels,
        title='Sector (n = unique POIs)',
        loc='center left',
        bbox_to_anchor=(1, 0, 0.5, 1),
        fontsize=9,
    )
    ax.set_title(
        f'Foot Traffic by Sector: {location_name}\n'
        f'n = {total_pois:,} total unique POIs  |  {total_visits:,} total visits',
        fontsize=13, fontweight='bold',
    )
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    log_step(f"🥧 NAICS pie chart → {os.path.basename(save_path)}")


print("✅ Cell 1 complete — imports, config, spatial helpers, chart functions loaded.")


✅  WEEKLY_DATA_FOLDER: /primary/work/ice/temp/california-foot-traffic-weekly-patterns-plus/california-weekly-data-all/
✅  LATINO_TRACTS_GEOJSON: /primary/work/ice/inputs/LACHD_majority_latino_neighborhoods_census_tracts_2020.geojson
✅  HALF_MILE_GEOJSON: /primary/work/ice/inputs/ice-june-6-2025-one-week-half_mile_buffers.geojson
✅  QUARTER_MILE_GEOJSON: /primary/work/ice/inputs/ice-june-6-2025-one-week-quarter_mile_buffers.geojson
✅  OUT_DIR: /primary/work/ice/outputs/raid_analysis
✅ Cell 1 complete — imports, config, spatial helpers, chart functions loaded.


In [30]:
def create_poi_bar_chart(results_list, save_path=None):
    """Bar chart comparing average weekly POI counts (n) in 2024 vs 2025 for selected buffers.

    Parameters
    ----------
    results_list : list of (location_name, area_df) tuples
        Each area_df is the DataFrame returned by spatial_aggregate (must have
        'week_start' and 'n_pois' columns).
    save_path : str or None
        If provided, saves the chart to this path.
    """
    if not results_list:
        return

    names, counts_2024, counts_2025 = [], [], []
    for location_name, area_df in results_list:
        if area_df is None or area_df.empty or 'n_pois' not in area_df.columns:
            continue
        df_2024 = area_df[area_df['week_start'].dt.year == 2024]['n_pois']
        df_2025 = area_df[area_df['week_start'].dt.year == 2025]['n_pois']
        short = (location_name
                 .replace('half_mile_', '0.5mi ')
                 .replace('quarter_mile_', '0.25mi '))
        names.append(short)
        counts_2024.append(float(df_2024.mean()) if not df_2024.empty else 0.0)
        counts_2025.append(float(df_2025.mean()) if not df_2025.empty else 0.0)

    if not names:
        log_step("⚠️ No POI n-data available for bar chart")
        return

    x = np.arange(len(names))
    width = 0.35
    fig, ax = plt.subplots(figsize=(max(10, len(names) * 2.2), 7))

    bars_2024 = ax.bar(x - width / 2, counts_2024, width,
                       label='2024 Avg Weekly POIs', color='#1f77b4', alpha=0.85)
    bars_2025 = ax.bar(x + width / 2, counts_2025, width,
                       label='2025 Avg Weekly POIs', color='#d62728', alpha=0.85)

    ax.bar_label(bars_2024, fmt='%.0f', padding=3, fontsize=9)
    ax.bar_label(bars_2025, fmt='%.0f', padding=3, fontsize=9)

    ax.set_ylabel('Average Weekly Unique POIs (n)', fontsize=13)
    ax.set_title('Average Weekly POI Count: 2024 vs 2025\nSelected Buffer Locations',
                 fontsize=16, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=30, ha='right', fontsize=10)
    ax.legend(fontsize=11)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, p: f'{int(v):,}'))
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    log_step("📊 POI bar chart (2024 vs 2025) saved.")


In [31]:

# ============================================
# 📐 STATS HELPERS
# ============================================

try:
    from scipy import stats as scipy_stats
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

RAID_DATE = pd.Timestamp('2025-06-05')
N_LAGS    = 2   # ±2 weeks around raid date (2-week analysis window)


def _safe_ttest(a, b):
    if not HAS_SCIPY or len(a) < 2 or len(b) < 2:
        return float('nan'), float('nan')
    try:
        t, p = scipy_stats.ttest_ind(a, b, equal_var=False)
        return t, p
    except Exception:
        return float('nan'), float('nan')


def _sig_label(p):
    if pd.isna(p):   return 'n/a'
    if p < 0.001:    return '***'
    if p < 0.01:     return '**'
    if p < 0.05:     return '*'
    return 'ns'


def _fmt(v, pct=False):
    if pd.isna(v):
        return 'N/A'
    if pct:
        return f'{v:+.1f}%'
    return f'{v:,.0f}'


def parse_raid_date(source):
    """Return datetime for event. Accepts dict-like or object with Date attribute."""
    if source is None:
        return None
    candidates = []
    if isinstance(source, dict):
        candidates = ['raid_date', 'Raid_Date', 'Raid Date', 'Date', 'date']
        for key in candidates:
            if key in source and pd.notna(source[key]):
                return pd.to_datetime(source[key], errors='coerce')
    else:
        for key in ['raid_date', 'Raid_Date', 'Raid Date', 'Date', 'date']:
            value = getattr(source, key, None)
            if pd.notna(value):
                return pd.to_datetime(value, errors='coerce')
    return None


# ============================================
# 📊 STATS CSV FUNCTIONS
# ============================================

def generate_descriptive_stats_csv(la_df, area_df, location_name, radius_label, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    rows = []
    for label, df in [('LA County', la_df), (f'{radius_label} Buffer', area_df)]:
        if df.empty:
            continue
        rows.append({
            'Series': label,
            'N_weeks': len(df),
            'Mean': df['total_visits'].mean(),
            'Median': df['total_visits'].median(),
            'Std': df['total_visits'].std(),
            'Min': df['total_visits'].min(),
            'Max': df['total_visits'].max(),
            'Date_Start': df['week_start'].min().date(),
            'Date_End': df['week_start'].max().date(),
        })
    if rows:
        pd.DataFrame(rows).to_csv(
            os.path.join(save_dir, f'descriptive_stats_{location_name}.csv'), index=False)
        log_step(f"📊 Descriptive stats → {location_name}")


def generate_pre_post_stats_csv(la_df, area_df, location_name, radius_label, save_dir, raid_date=None):
    os.makedirs(save_dir, exist_ok=True)
    rows = []
    raid_date = pd.to_datetime(raid_date) if raid_date is not None else RAID_DATE
    if pd.isna(raid_date):
        raid_date = RAID_DATE
    pre_window = (raid_date - pd.Timedelta(weeks=2), raid_date)
    post_window = (raid_date, raid_date + pd.Timedelta(weeks=2))
    for label, df in [('LA County', la_df), (f'{radius_label} Buffer', area_df)]:
        if df.empty:
            continue
        pre  = df[(df['week_start'] >= pre_window[0]) & (df['week_start'] < pre_window[1])]['total_visits']
        post = df[(df['week_start'] > post_window[0]) & (df['week_start'] <= post_window[1])]['total_visits']
        pct_change = ((post.mean() / pre.mean()) - 1) * 100 if (len(pre) and pre.mean() != 0) else float('nan')
        _, p = _safe_ttest(pre.values, post.values)
        rows.append({
            'Series': label,
            'Pre_Raid_Mean': pre.mean(),
            'Post_Raid_Mean': post.mean(),
            'Pct_Change': pct_change,
            'P_Value': p,
            'Significance': _sig_label(p),
        })
    if rows:
        pd.DataFrame(rows).to_csv(
            os.path.join(save_dir, f'pre_post_stats_{location_name}.csv'), index=False)
        log_step(f"📊 Pre/post stats → {location_name}")


def generate_yoy_stats_csv(la_df, area_df, location_name, radius_label, save_dir, raid_date=None):
    os.makedirs(save_dir, exist_ok=True)
    rows = []
    raid_date = pd.to_datetime(raid_date) if raid_date is not None else RAID_DATE
    if pd.isna(raid_date):
        raid_date = RAID_DATE
    for label, df in [('LA County', la_df), (f'{radius_label} Buffer', area_df)]:

        if df.empty:
            continue
        # June window: 4 weeks post-raid
        post_25 = df[(df['week_start'] >= raid_date) &
                     (df['week_start'] < raid_date + pd.Timedelta(weeks=4))]['total_visits']
        raid_py = raid_date - pd.DateOffset(years=1)
        post_24 = df[(df['week_start'] >= raid_py) &
                     (df['week_start'] < raid_py + pd.Timedelta(weeks=4))]['total_visits']
        pct = ((post_25.mean() / post_24.mean()) - 1) * 100 if (len(post_24) and post_24.mean() != 0) else float('nan')
        _, p = _safe_ttest(post_25.values, post_24.values)
        rows.append({
            'Series': label,
            '2025_PostRaid_Mean': post_25.mean(),
            '2024_SamePeriod_Mean': post_24.mean(),
            'YoY_Pct_Change': pct,
            'P_Value': p,
            'Significance': _sig_label(p),
        })
    if rows:
        pd.DataFrame(rows).to_csv(
            os.path.join(save_dir, f'yoy_stats_{location_name}.csv'), index=False)
        log_step(f"📊 YoY stats → {location_name}")


# ============================================
# 📈 LAG / EVENT STUDY
# ============================================

def _expand_weekly_to_daily(df):
    """Convert weekly total visits (per week_start) into a daily interpolated series."""
    if df.empty or 'week_start' not in df.columns or 'total_visits' not in df.columns:
        return pd.DataFrame(columns=['date', 'total_visits', 'n_pois'])

    weekly = df[['week_start', 'total_visits']].dropna(subset=['week_start']).copy()
    if weekly.empty:
        return pd.DataFrame(columns=['date', 'total_visits', 'n_pois'])

    weekly = weekly.sort_values('week_start')
    start = weekly['week_start'].min()
    end = weekly['week_start'].max() + pd.Timedelta(days=6)
    daily_index = pd.date_range(start, end, freq='D')

    series = pd.Series(weekly['total_visits'].values, index=weekly['week_start'])
    series = series.reindex(daily_index.union(series.index)).sort_index()
    series = series.interpolate(method='time', limit_direction='both')
    series = series.reindex(daily_index)

    daily = pd.DataFrame({'date': series.index, 'total_visits': series.values})

    if 'n_pois' in df.columns:
        nseries = pd.Series(df.set_index('week_start')['n_pois']).reindex(daily_index.union(df['week_start'])).sort_index()
        nseries = nseries.interpolate(method='time', limit_direction='both')
        nseries = nseries.reindex(daily_index)
        daily['n_pois'] = nseries.round().astype('Int64')
    else:
        daily['n_pois'] = pd.NA

    return daily


def _get_event_df(df, anchor_date, n_lags=N_LAGS):
    """Return DataFrame with columns [event_week, event_date, total_visits, n_pois]."""
    if df.empty:
        return pd.DataFrame(columns=['event_week', 'event_date', 'total_visits', 'n_pois'])
    has_n  = 'n_pois' in df.columns
    result = []
    for lag in range(-n_lags, n_lags + 1):
        target = anchor_date + pd.Timedelta(weeks=lag)
        diffs  = (df['week_start'] - target).abs()
        i      = diffs.idxmin()
        if diffs[i] <= pd.Timedelta(days=4):
            chosen_date = df.loc[i, 'week_start']
            row = {
                'event_week': lag,
                'event_date': chosen_date,
                'total_visits': df.loc[i, 'total_visits']
            }
            row['n_pois'] = int(df.loc[i, 'n_pois']) if has_n and pd.notna(df.loc[i, 'n_pois']) else None
            result.append(row)
    return pd.DataFrame(result)


def _get_event_df_daily(df, anchor_date, n_days=14):
    """Return DataFrame with daily event points around anchor_date."""
    if df.empty:
        return pd.DataFrame(columns=['event_day', 'event_date', 'total_visits', 'n_pois'])

    daily_df = _expand_weekly_to_daily(df)
    if daily_df.empty:
        return pd.DataFrame(columns=['event_day', 'event_date', 'total_visits', 'n_pois'])

    min_date = anchor_date - pd.Timedelta(days=n_days)
    max_date = anchor_date + pd.Timedelta(days=n_days)
    window = daily_df[(daily_df['date'] >= min_date) & (daily_df['date'] <= max_date)].copy()
    if window.empty:
        return pd.DataFrame(columns=['event_day', 'event_date', 'total_visits', 'n_pois'])

    window['event_day'] = (window['date'] - anchor_date).dt.days
    window = window.rename(columns={'date': 'event_date'})
    return window[['event_day', 'event_date', 'total_visits', 'n_pois']]


def _normalize(evt_df):
    """Index visits to pre-event mean = 100 for weekly or daily event frames."""
    if 'event_week' in evt_df.columns:
        pre = evt_df[evt_df['event_week'] < 0]['total_visits']
    elif 'event_day' in evt_df.columns:
        pre = evt_df[evt_df['event_day'] < 0]['total_visits']
    else:
        pre = pd.Series([], dtype=float)

    base = pre.mean() if not pre.empty else None
    if not base:
        return evt_df.assign(indexed=evt_df['total_visits'])
    return evt_df.assign(indexed=evt_df['total_visits'] / base * 100)


def generate_lag_analysis(la_df, area_df, location_name, radius_label, save_dir, raid_date=None, n_lags=N_LAGS):
    if la_df.empty or area_df.empty:
        log_step(f"⚠️ Skipping lag analysis for {location_name}: insufficient data")
        return

    raid_date = pd.to_datetime(raid_date) if raid_date is not None else RAID_DATE
    if pd.isna(raid_date):
        raid_date = RAID_DATE

    # event windows around raid_date
    area_evt = _get_event_df(area_df, raid_date,   n_lags)
    la_evt   = _get_event_df(la_df,   raid_date,   n_lags)
    area_evt_daily = _get_event_df_daily(area_df, raid_date, n_days=n_lags * 7)
    la_evt_daily   = _get_event_df_daily(la_df,   raid_date, n_days=n_lags * 7)

    # 1 year prior baseline
    raid_py     = raid_date - pd.DateOffset(years=1)
    area_evt_py = _get_event_df(area_df, raid_py,  n_lags)
    la_evt_py   = _get_event_df(la_df,   raid_py,  n_lags)
    area_evt_py_daily = _get_event_df_daily(area_df, raid_py, n_days=n_lags * 7)
    la_evt_py_daily   = _get_event_df_daily(la_df,   raid_py, n_days=n_lags * 7)

    if (area_evt.empty and la_evt.empty and
        area_evt_daily.empty and la_evt_daily.empty):
        log_step(f"⚠️ No event data for {location_name}")
        return

    # ── Save lag CSV (weekly) ──
    rows = []
    for lag in range(-n_lags, n_lags + 1):
        row_d = {
            'event_week': lag,
            'event_date': raid_date + pd.Timedelta(weeks=lag)
        }
        for key, ev in [('area_2025', area_evt), ('la_2025', la_evt),
                        ('area_2024', area_evt_py), ('la_2024', la_evt_py)]:
            match = ev[ev['event_week'] == lag]
            row_d[f'{key}_visits'] = match['total_visits'].values[0] if len(match) else None
            n_col = match['n_pois'].values[0] if (len(match) and 'n_pois' in match.columns) else None
            row_d[f'{key}_n_pois'] = int(n_col) if n_col is not None and pd.notna(n_col) else None
        rows.append(row_d)
    lag_df = pd.DataFrame(rows)
    lag_csv = os.path.join(save_dir, f'lag_analysis_results_{location_name}.csv')
    lag_df.to_csv(lag_csv, index=False)
    log_step(f"📋 Lag analysis table exported: {os.path.basename(lag_csv)}")

    # ── Save daily lag CSV (interpolated sub-week) ──
    if not area_evt_daily.empty or not la_evt_daily.empty:
        df_daily = pd.DataFrame()
        if not area_evt_daily.empty:
            df_daily = area_evt_daily[['event_day', 'event_date', 'total_visits']].rename(
                columns={'total_visits': 'area_2025_visits'})
        if not la_evt_daily.empty:
            la_df_daily = la_evt_daily[['event_day', 'event_date', 'total_visits']].rename(
                columns={'total_visits': 'la_2025_visits'})
            if df_daily.empty:
                df_daily = la_df_daily
            else:
                df_daily = pd.merge(df_daily, la_df_daily, on=['event_day', 'event_date'], how='outer')
        if not area_evt_py_daily.empty:
            area_2024 = area_evt_py_daily[['event_day', 'event_date', 'total_visits']].rename(
                columns={'total_visits': 'area_2024_visits'})
            df_daily = pd.merge(df_daily, area_2024, on=['event_day', 'event_date'], how='outer')
        if not la_evt_py_daily.empty:
            la_2024 = la_evt_py_daily[['event_day', 'event_date', 'total_visits']].rename(
                columns={'total_visits': 'la_2024_visits'})
            df_daily = pd.merge(df_daily, la_2024, on=['event_day', 'event_date'], how='outer')

        df_daily = df_daily.sort_values(['event_day', 'event_date']).reset_index(drop=True)
        daily_csv = os.path.join(save_dir, f'lag_analysis_daily_{location_name}.csv')
        df_daily.to_csv(daily_csv, index=False)
        log_step(f"📋 Daily lag analysis table exported: {os.path.basename(daily_csv)}")

    # ── Display raw-traffic table inline ──────────────────────────────────────
    print(f"\n📊 Raw Weekly Traffic — Event Study: {location_name}  (week 0 = raid date)")
    display_cols_2025 = ['event_week', 'event_date', 'area_2025_visits', 'area_2025_n_pois',
                         'la_2025_visits', 'la_2025_n_pois']
    display_cols_all  = display_cols_2025 + ['area_2024_visits', 'area_2024_n_pois',
                                              'la_2024_visits',  'la_2024_n_pois']
    disp_cols = [c for c in display_cols_all if c in lag_df.columns]
    fmt_dict  = {c: '{:,.0f}' for c in disp_cols if c.endswith('_visits') or c.endswith('_n_pois')}
    display(lag_df[disp_cols].style
            .format(fmt_dict, na_rep='—')
            .set_caption(f'Event-study raw traffic — {location_name}  '
                         f'(columns ending _n_pois = unique POIs that week)')
            .hide(axis='index'))

    # ── Single overlay chart: 2025 solid + 2024 faded on same axes (event-day relative) ──
    fig, ax = plt.subplots(figsize=(12, 7))

    # Use daily event-study interpolation when available, otherwise fallback to weekly points.
    plot_series = []
    if not area_evt_daily.empty:
        series = area_evt_daily.copy()
        series['relative_day'] = series['event_day']
        plot_series.append((series, f'{radius_label} Buffer (2025 daily)', '#d62728', 's', 1.0, 2.5, '-'))
    elif not area_evt.empty:
        series = area_evt.copy()
        series['relative_day'] = series['event_week'] * 7
        plot_series.append((series, f'{radius_label} Buffer (2025 weekly)', '#d62728', 's', 1.0, 2.5, '-'))

    if not la_evt_daily.empty:
        series = la_evt_daily.copy()
        series['relative_day'] = series['event_day']
        plot_series.append((series, 'LA County (2025 daily)', '#1f77b4', 'o', 1.0, 2.5, '-'))
    elif not la_evt.empty:
        series = la_evt.copy()
        series['relative_day'] = series['event_week'] * 7
        plot_series.append((series, 'LA County (2025 weekly)', '#1f77b4', 'o', 1.0, 2.5, '-'))

    if not area_evt_py_daily.empty:
        series = area_evt_py_daily.copy()
        series['relative_day'] = series['event_day']
        plot_series.append((series, f'{radius_label} Buffer (2024 daily)', '#d62728', 's', 0.35, 1.5, '--'))
    elif not area_evt_py.empty:
        series = area_evt_py.copy()
        series['relative_day'] = series['event_week'] * 7
        plot_series.append((series, f'{radius_label} Buffer (2024 weekly)', '#d62728', 's', 0.35, 1.5, '--'))

    if not la_evt_py_daily.empty:
        series = la_evt_py_daily.copy()
        series['relative_day'] = series['event_day']
        plot_series.append((series, 'LA County (2024 daily)', '#1f77b4', 'o', 0.35, 1.5, '--'))
    elif not la_evt_py.empty:
        series = la_evt_py.copy()
        series['relative_day'] = series['event_week'] * 7
        plot_series.append((series, 'LA County (2024 weekly)', '#1f77b4', 'o', 0.35, 1.5, '--'))

    for evt_data, lbl, color, mkr, alpha, lw, ls in plot_series:
        if not evt_data.empty:
            normed = _normalize(evt_data)
            if 'n_pois' in evt_data.columns and not evt_data['n_pois'].dropna().empty:
                n_val = int(evt_data['n_pois'].dropna().mean())
                lbl = f"{lbl} (n={n_val:,})"
            ax.plot(normed['relative_day'], normed['indexed'], color=color, lw=lw,
                    marker=mkr, markersize=6 if alpha < 1 else 8,
                    alpha=alpha, label=lbl, ls=ls)

    ax.axvline(0, color='red', ls='--', lw=1.8, alpha=0.7, label='Raid Date (day 0)')
    ax.axhline(100, color='gray', ls=':', lw=1, alpha=0.4)
    ax.set_xlabel('Days relative to raid (−14 to +14)', fontsize=12)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))
    ax.set_xlim(-n_lags * 7, n_lags * 7)
    plt.xticks(rotation=0)
    ax.set_ylabel('Indexed Visits (Pre-Raid Mean = 100)', fontsize=12)
    ax.set_title(f'Lag Analysis: {location_name}\n2025 (solid) vs 2024 Baseline (faded)',
                 fontsize=14, fontweight='bold')
    ax.legend(fontsize=10, loc='lower left')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(save_dir, f'lag_plot_{location_name}.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    log_step(f"📈 Lag plot → {os.path.basename(plot_path)}")


# ============================================
# 📊 2-WEEK PRE/POST RAID COMPARISON
# ============================================

def generate_2week_prepost(la_df, area_df, location_name, radius_label, save_dir, raid_id=None, raid_date=None):
    """Avg traffic for 2 weeks immediately before vs 2 weeks immediately after the raid.

    Weeks used:
      Before : event_week -2 and -1  (two calendar weeks ending just before raid)
      Raid   : event_week  0          (the raid week itself)
      After  : event_week +1 and +2  (two calendar weeks starting just after raid)

    Outputs a CSV and displays a styled table inline.
    """
    os.makedirs(save_dir, exist_ok=True)
    raid_date = pd.to_datetime(raid_date) if raid_date is not None else RAID_DATE
    if pd.isna(raid_date):
        raid_date = RAID_DATE
    area_evt = _get_event_df(area_df, raid_date, n_lags=2)
    la_evt   = _get_event_df(la_df,   raid_date, n_lags=2)

    rows = []
    for label, evt in [('Area Buffer', area_evt), ('LA County', la_evt)]:
        pre_rows  = evt[evt['event_week'].isin([-2, -1])]
        post_rows = evt[evt['event_week'].isin([1,   2])]
        raid_row  = evt[evt['event_week'] == 0]

        pre_avg  = pre_rows['total_visits'].mean()  if not pre_rows.empty  else float('nan')
        post_avg = post_rows['total_visits'].mean() if not post_rows.empty else float('nan')
        raid_val = raid_row['total_visits'].values[0] if not raid_row.empty else float('nan')

        # n_pois: average unique POIs across the four comparison weeks
        has_n = 'n_pois' in evt.columns
        pre_n  = pre_rows['n_pois'].mean()  if (has_n and not pre_rows.empty)  else None
        post_n = post_rows['n_pois'].mean() if (has_n and not post_rows.empty) else None

        diff    = post_avg - pre_avg
        pct_chg = (diff / pre_avg * 100) if (not pd.isna(pre_avg) and pre_avg != 0) else float('nan')

        raid_buffer = f"{raid_id}_{radius_label}" if raid_id is not None else radius_label
        rows.append({
            'Raid_ID':         raid_id,
            'Buffer_Range':    radius_label,
            'Raid_Buffer':     raid_buffer,
            'Series':          f'{label} ({radius_label})',
            'Avg_2wk_Before':  pre_avg,
            'Avg_n_Before':    round(pre_n,  1) if pre_n  is not None else None,
            'Raid_Week':       raid_val,
            'Avg_2wk_After':   post_avg,
            'Avg_n_After':     round(post_n, 1) if post_n is not None else None,
            'Raw_Difference':  diff,
            'Pct_Change':      pct_chg,
        })

    result_df = pd.DataFrame(rows)
    csv_path  = os.path.join(save_dir, f'2week_prepost_{location_name}.csv')
    result_df.to_csv(csv_path, index=False)

    print(f"\n📊 2-Week Pre/Post Raid Traffic: {location_name}  (raid date: {raid_date.date()})")
    fmt = {
        'Avg_2wk_Before': '{:,.0f}', 'Avg_n_Before':  '{:.1f}',
        'Raid_Week':       '{:,.0f}',
        'Avg_2wk_After':  '{:,.0f}', 'Avg_n_After':   '{:.1f}',
        'Raw_Difference': '{:+,.0f}', 'Pct_Change':    '{:+.1f}%',
    }
    # Only format columns that are present and numeric
    fmt_safe = {k: v for k, v in fmt.items() if k in result_df.columns}
    display(result_df.style.format(fmt_safe, na_rep='—').hide(axis='index'))
    log_step(f"📊 2-week pre/post → {os.path.basename(csv_path)}")
    return result_df


# ============================================
# 📋 OVERVIEW TABLE
# ============================================

def generate_overview_table(la_df, area_df, location_name, radius_label, save_dir, raid_id=None, raid_date=None):
    os.makedirs(save_dir, exist_ok=True)
    if la_df.empty or area_df.empty:
        log_step(f"⚠️ Skipping overview table for {location_name}")
        return

    raid_date = pd.to_datetime(raid_date) if raid_date is not None else RAID_DATE
    if pd.isna(raid_date):
        raid_date = RAID_DATE

    pre_window = (raid_date - pd.Timedelta(weeks=2), raid_date)
    post_window = (raid_date, raid_date + pd.Timedelta(weeks=2))
    pre_covid_cutoff = pd.Timestamp('2020-03-01')

    pre_area_df = area_df[(area_df['week_start'] >= pre_window[0]) & (area_df['week_start'] < pre_window[1])]
    post_area_df = area_df[(area_df['week_start'] > post_window[0]) & (area_df['week_start'] <= post_window[1])]
    pre_la_df = la_df[(la_df['week_start'] >= pre_window[0]) & (la_df['week_start'] < pre_window[1])]
    post_la_df = la_df[(la_df['week_start'] > post_window[0]) & (la_df['week_start'] <= post_window[1])]

    pre_area = pre_area_df['total_visits']
    post_area = post_area_df['total_visits']
    pre_la = pre_la_df['total_visits']
    post_la = post_la_df['total_visits']

    pre_area_n = pre_area_df['n_pois'].mean() if 'n_pois' in pre_area_df.columns and not pre_area_df.empty else float('nan')
    post_area_n = post_area_df['n_pois'].mean() if 'n_pois' in post_area_df.columns and not post_area_df.empty else float('nan')
    pre_la_n = pre_la_df['n_pois'].mean() if 'n_pois' in pre_la_df.columns and not pre_la_df.empty else float('nan')
    post_la_n = post_la_df['n_pois'].mean() if 'n_pois' in post_la_df.columns and not post_la_df.empty else float('nan')

    pre_covid_area = area_df[area_df['week_start'] < pre_covid_cutoff]['total_visits']
    pre_covid_la = la_df[la_df['week_start'] < pre_covid_cutoff]['total_visits']

    pct_area = ((post_area.mean() / pre_area.mean()) - 1) * 100 if (len(pre_area) and pre_area.mean() != 0) else float('nan')
    pct_la = ((post_la.mean() / pre_la.mean()) - 1) * 100 if (len(pre_la) and pre_la.mean() != 0) else float('nan')
    _, p = _safe_ttest(pre_area.values, post_area.values)

    # YoY
    raid_py = raid_date - pd.DateOffset(years=1)
    post_py = area_df[(area_df['week_start'] >= raid_py) &
                      (area_df['week_start'] < raid_py + pd.Timedelta(weeks=4))]['total_visits']
    post_25_4w = area_df[(area_df['week_start'] >= raid_date) &
                         (area_df['week_start'] < raid_date + pd.Timedelta(weeks=4))]['total_visits']
    yoy_pct = ((post_25_4w.mean() / post_py.mean()) - 1) * 100 if (len(post_py) and post_py.mean() != 0) else float('nan')

    # Keep existing table as it is, but reuse added rows for CSV output.
    area_row = {
        'Raid_ID':               raid_id if raid_id is not None else '',
        'Raid_Date':             raid_date.date() if not pd.isna(raid_date) else '',
        'Analysis_Type':         'Overall',
        'Buffer_Distance':       radius_label,
        'Series':                f'{radius_label} Buffer',
        'Pre_Raid_Mean':         pre_area.mean(),
        'Post_Raid_Mean':        post_area.mean(),
        'Pre_Raid_n':            pre_area_n,
        'Post_Raid_n':           post_area_n,
        'LA_Pre_Raid_n':         pre_la_n,
        'LA_Post_Raid_n':        post_la_n,
        'Pre_COVID_Mean':        pre_covid_area.mean(),
        'LA_Pre_COVID_Mean':     pre_covid_la.mean(),
        'Pct_Change':            pct_area,
        'Statistical_Significance': _sig_label(p),
        'YoY_Pct_Change':        yoy_pct,
        'Vs_LA_Pct_Change':      pct_area - pct_la if not (pd.isna(pct_area) or pd.isna(pct_la)) else float('nan'),
    }
    la_ratio_p = _safe_ttest(pre_la.values, post_la.values)[1] if not pre_la.empty and not post_la.empty else float('nan')
    la_row = {
        'Raid_ID':               raid_id if raid_id is not None else '',
        'Raid_Date':             raid_date.date() if not pd.isna(raid_date) else '',
        'Analysis_Type':         'Overall',
        'Buffer_Distance':       radius_label,
        'Series':                'LA County',
        'Pre_Raid_Mean':         pre_la.mean(),
        'Post_Raid_Mean':        post_la.mean(),
        'Pre_Raid_n':            pre_la_n,
        'Post_Raid_n':           post_la_n,
        'LA_Pre_Raid_n':         pre_la_n,
        'LA_Post_Raid_n':        post_la_n,
        'Pre_COVID_Mean':        pre_covid_la.mean(),
        'LA_Pre_COVID_Mean':     pre_covid_la.mean(),
        'Pct_Change':            pct_la,
        'Statistical_Significance': _sig_label(la_ratio_p),
        'YoY_Pct_Change':        np.nan,
        'Vs_LA_Pct_Change':      np.nan,
    }

    rows = [area_row, la_row]
    csv_path = os.path.join(save_dir, f'analysis_overview_table_{location_name}.csv')
    pd.DataFrame(rows).to_csv(csv_path, index=False)

    # ── Visual table ──
    headers = ['Raid ID', 'Raid Date', 'Analysis', 'Buffer', 'Pre-Raid\nMean', 'Post-Raid\nMean',
               'Pre-Raid\nN', 'Post-Raid\nN', 'LA Pre-Raid\nN', 'LA Post-Raid\nN',
               'Pre-COVID\nMean', 'YoY %\nChange', '% Change', 'Sig']
    table_data = [[
        str(raid_id) if raid_id is not None else '',
        raid_date.date().isoformat() if not pd.isna(raid_date) else '',
        'Overall', radius_label,
        _fmt(pre_area.mean()), _fmt(post_area.mean()),
        _fmt(pre_area_n), _fmt(post_area_n), _fmt(pre_la_n), _fmt(post_la_n),
        _fmt(pre_covid_area.mean()),
        _fmt(yoy_pct, pct=True), _fmt(pct_area, pct=True), _sig_label(p),
    ]]  
    fig, ax = plt.subplots(figsize=(20, max(2, 0.6 * len(table_data) + 1.2)))
    ax.axis('off')
    tbl = ax.table(cellText=table_data, colLabels=headers, cellLoc='center', loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.8)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor('#2c3e50')
            cell.set_text_props(color='white', fontweight='bold')
        elif r % 2 == 0:
            cell.set_facecolor('#f2f2f2')
        cell.set_edgecolor('#cccccc')
    ax.set_title(f'Analysis Overview: {location_name}', fontsize=14, fontweight='bold', pad=12)
    plt.tight_layout()
    img_path = os.path.join(save_dir, f'analysis_overview_table_{location_name}.png')
    plt.savefig(img_path, dpi=200, bbox_inches='tight')
    plt.show()
    plt.close()
    log_step(f"📋 Overview table → {os.path.basename(csv_path)}")


def extract_poi_details(weekly_files, area_gdf, area_label, raid_id=None, out_dir=None, raid_date=None):
    """Extract POI-level records (lat/lon, NAICS, date) from weekly data for this area.
    Filters to ±2 weeks around raid_date in 2025, and the equivalent window in 2024.
    """
    if out_dir is None:
        out_dir = OUT_DIR

    # ── Build ±2-week date windows for 2025 and 2024 ──
    if raid_date is not None:
        raid_date = pd.to_datetime(raid_date, errors='coerce')
    if raid_date is None or pd.isna(raid_date):
        raid_date = RAID_DATE
    window = pd.Timedelta(weeks=2)
    win_2025_start = raid_date - window
    win_2025_end   = raid_date + window
    # Same calendar window one year earlier
    try:
        win_2024_start = win_2025_start.replace(year=win_2025_start.year - 1)
        win_2024_end   = win_2025_end.replace(year=win_2025_end.year - 1)
    except ValueError:  # leap-day edge case
        win_2024_start = win_2025_start - pd.DateOffset(years=1)
        win_2024_end   = win_2025_end - pd.DateOffset(years=1)
    log_step(f"📅 POI window 2025: {win_2025_start.date()} → {win_2025_end.date()}")
    log_step(f"📅 POI window 2024: {win_2024_start.date()} → {win_2024_end.date()}")

    records = []

    for fp in weekly_files:
        try:
            cols = [
                'DATE_RANGE_START', 'VISITS_BY_DAY', 'LONGITUDE', 'LATITUDE', 'NAICS_CODE',
                # Advan POI identity
                'PERSISTENT_ID', 'LOCATION_NAME', 'BRAND',
                # Geography
                'POSTAL_CODE', 'STREET_ADDRESS', 'CITY', 'REGION',
                # Economic analysis fields
                'MEDIAN_DWELL', 'TOP_CATEGORY', 'SUB_CATEGORY', 'VISIT_COUNTS',
            ]
            for chunk_df in _read_csv_robust(fp, cols):
                if chunk_df.empty:
                    continue
                chunk_df = chunk_df.dropna(subset=['LONGITUDE', 'LATITUDE'])
                if chunk_df.empty:
                    continue
                chunk_df = chunk_df[
                    (chunk_df['LONGITUDE'].between(-180, 180)) &
                    (chunk_df['LATITUDE'].between(-90, 90))
                ]
                if chunk_df.empty:
                    continue
                # Derive week_start from DATE_RANGE_START
                chunk_df['DATE_RANGE_START'] = pd.to_datetime(chunk_df['DATE_RANGE_START'], errors='coerce')
                chunk_df['week_start'] = chunk_df['DATE_RANGE_START'].dt.normalize()
                gdf = gpd.GeoDataFrame(
                    chunk_df,
                    geometry=gpd.points_from_xy(chunk_df['LONGITUDE'], chunk_df['LATITUDE']),
                    crs='EPSG:4326'
                )
                gdf = gdf[gdf.geometry.is_valid]
                if gdf.empty:
                    continue
                try:
                    joined = gpd.sjoin(gdf, area_gdf[['geometry']].copy(), how='inner', predicate='within')
                except Exception as e:
                    log_step(f"⚠️ POI export spatial join failed: {str(e)[:80]}")
                    continue
                if joined.empty:
                    continue
                # Map Advan columns → output fields (exact names from schema inspection)
                poi_id_col   = next((c for c in ['PERSISTENT_ID', 'FOOTPRINT_ID', 'PERSISTENT_ID_STORE',
                                                  'placekey', 'safegraph_place_id'] if c in joined.columns), None)
                poi_name_col = next((c for c in ['LOCATION_NAME', 'BRAND',
                                                  'location_name', 'business_name'] if c in joined.columns), None)
                brand_col    = 'BRAND' if 'BRAND' in joined.columns else None
                zipcode_col  = next((c for c in ['POSTAL_CODE', 'postal_code', 'zipcode'] if c in joined.columns), None)

                for _, r in joined.iterrows():
                    ws = r['week_start']
                    in_2025 = win_2025_start <= ws <= win_2025_end
                    in_2024 = win_2024_start <= ws <= win_2024_end
                    if not (in_2025 or in_2024):
                        continue
                    yr = ws.year
                    records.append({
                        'raid_id': raid_id,
                        'location_name': area_label,
                        'week_start': r['week_start'],
                        'date': r['week_start'],
                        'year': yr,
                        'post_raid': 1 if yr == 2025 else 0,
                        'poi_lat': r['LATITUDE'],
                        'poi_lon': r['LONGITUDE'],
                        'poi_id': r.get(poi_id_col) if poi_id_col else None,
                        'poi_name': r.get(poi_name_col) if poi_name_col else None,
                        'brand': r.get(brand_col) if brand_col else None,
                        'zipcode': r.get(zipcode_col) if zipcode_col else None,
                        'street_address': r.get('STREET_ADDRESS'),
                        'city': r.get('CITY'),
                        'region': r.get('REGION'),
                        'naics_code': r.get('NAICS_CODE'),
                        'top_category': r.get('TOP_CATEGORY'),
                        'sub_category': r.get('SUB_CATEGORY'),
                        'median_dwell': r.get('MEDIAN_DWELL'),
                        'visits': parse_visits_fast(r.get('VISITS_BY_DAY', None)),
                        'visit_counts': r.get('VISIT_COUNTS'),
                    })
        except Exception as e:
            log_step(f"⚠️ POI export failed file {os.path.basename(fp)}: {str(e)[:80]}")
            continue

    if not records:
        return None

    poi_df = pd.DataFrame(records)
    poi_df['week_start'] = pd.to_datetime(poi_df['week_start'])
    out_csv = os.path.join(out_dir, f'poi_details_{area_label}.csv')
    poi_df.to_csv(out_csv, index=False)
    log_step(f"📋 POI details CSV exported: {os.path.basename(out_csv)}")

    # output GeoJSON as well
    try:
        geo_df = gpd.GeoDataFrame(
            poi_df,
            geometry=gpd.points_from_xy(poi_df['poi_lon'], poi_df['poi_lat']),
            crs='EPSG:4326'
        )
        out_geojson = os.path.join(out_dir, f'poi_details_{area_label}.geojson')
        geo_df.to_file(out_geojson, driver='GeoJSON')
        log_step(f"🗺️ POI details GeoJSON exported: {os.path.basename(out_geojson)}")
    except Exception as e:
        log_step(f"⚠️ Could not write POI GeoJSON: {str(e)[:80]}")
    return out_csv

print("✅ Cell 2 complete — stats functions, lag analysis, and overview table loaded.")

✅ Cell 2 complete — stats functions, lag analysis, and overview table loaded.


In [32]:

# ============================================
# 🎯 EXECUTION FUNCTIONS
# ============================================

def run_single_area(area_gdf, location_name, loc_cache, loc_stats, loc_out_dir,
                    high_impact_include_ag=True, raid_id=None, raid_date=None):
    """Run full analysis for a single buffer polygon area."""
    if area_gdf.empty:
        log_step(f"❌ Skipping {location_name}: area empty")
        return

    if raid_date is None:
        raid_date = parse_raid_date(area_gdf.iloc[0].to_dict()) if not area_gdf.empty else None
    raid_date = pd.to_datetime(raid_date, errors='coerce')
    if pd.isna(raid_date):
        raid_date = RAID_DATE

    all_files   = glob.glob(os.path.join(WEEKLY_DATA_FOLDER, '*.csv*'))
    valid_files = [f for f in all_files if os.path.getsize(f) > 5000]

    if not valid_files:
        log_step(f"⚠️  No weekly CSV files found in {WEEKLY_DATA_FOLDER}")
        log_step(f"   Set env var ICE_INPUT_BASE to the folder containing the Dewey downloads.")
        return

    high_naics  = get_naics_codes(high_impact_include_ag)
    high_suffix = "ag_incl" if high_impact_include_ag else "no_ag"
    high_label  = (f'{location_name} High Impact (+Ag)' if high_impact_include_ag
                   else f'{location_name} High Impact (No Ag)')

    la_tracts = gpd.read_file(LATINO_TRACTS_GEOJSON).to_crs('EPSG:4326')

    # NAICS breakdown is collected for free during the overall (unfiltered) passes
    la_overall, n_la_overall, la_naics = spatial_aggregate(
        valid_files, la_tracts, 'LA County', None,
        os.path.join(CACHE_DIR, 'la_overall.pkl'),
        collect_naics=True,
        naics_cache_file=os.path.join(CACHE_DIR, 'la_naics.pkl'),
    )
    area_overall, n_area_overall, area_naics = spatial_aggregate(
        valid_files, area_gdf, location_name, None,
        os.path.join(loc_cache, 'area_overall.pkl'),
        collect_naics=True,
        naics_cache_file=os.path.join(loc_cache, 'area_naics.pkl'),
    )
    la_high,   n_la_high,   _ = spatial_aggregate(valid_files, la_tracts, 'LA High Impact', high_naics,  os.path.join(CACHE_DIR, f'la_high_{high_suffix}.pkl'))
    area_high, n_area_high, _ = spatial_aggregate(valid_files, area_gdf,  high_label,       high_naics,  os.path.join(loc_cache, f'area_high_{high_suffix}.pkl'))

    high_subtitle = 'Ag+Grocery+Rest+Retail' if high_impact_include_ag else 'Grocery+Rest+Retail'

    print(f"\n{'='*60}")
    print(f"📊 Charts for: {location_name}")
    print(f"{'='*60}")

    create_single_chart(la_overall,   'LA County Overall',  '',       '#1f77b4', f'{loc_out_dir}/la_overall.png',
                        n_pois=n_la_overall)
    create_single_chart(area_overall, f'{location_name} Overall', '', '#d62728', f'{loc_out_dir}/location_overall.png',
                        n_pois=n_area_overall)
    create_comparison_chart(la_overall, area_overall, None, location_name, f'{loc_out_dir}/comp_overall.png',
                             n_pois_la=n_la_overall, n_pois_area=n_area_overall)

    create_single_chart(la_high,   'LA High Impact',  high_subtitle, '#2E8B57', f'{loc_out_dir}/la_high_{high_suffix}.png',
                        n_pois=n_la_high)
    create_single_chart(area_high, high_label,        high_subtitle, '#DC143C', f'{loc_out_dir}/location_high_{high_suffix}.png',
                        n_pois=n_area_high)
    create_comparison_chart(la_high, area_high, None, location_name, f'{loc_out_dir}/comp_high_{high_suffix}.png', high_subtitle,
                             n_pois_la=n_la_high, n_pois_area=n_area_high)

    create_2025_comparison_chart(la_overall, area_overall, location_name,
                                 f'{loc_out_dir}/comp_2025_overall.png',
                                 n_pois_la=n_la_overall, n_pois_area=n_area_overall)
    create_2025_comparison_chart(la_high, area_high, location_name,
                                 f'{loc_out_dir}/comp_2025_high_{high_suffix}.png', high_subtitle,
                                 n_pois_la=n_la_high, n_pois_area=n_area_high)

    generate_stats(la_overall, area_overall, f'{location_name}_overall', location_name)
    generate_stats(la_high,    area_high,    f'{location_name}_high_{high_suffix}', high_label)

    radius_label = '0.5mi' if 'half_mile' in location_name else '0.25mi'
    generate_descriptive_stats_csv(la_overall, area_overall, location_name, radius_label, loc_stats)
    generate_pre_post_stats_csv(la_overall, area_overall, location_name, radius_label, loc_stats, raid_date=raid_date)
    generate_yoy_stats_csv(la_overall, area_overall, location_name, radius_label, loc_stats, raid_date=raid_date)
    generate_lag_analysis(la_overall, area_overall, location_name, radius_label, loc_out_dir, raid_date=raid_date)
    twoweek_df = generate_2week_prepost(la_overall, area_overall, location_name, radius_label,
                                         loc_stats, raid_id=raid_id, raid_date=raid_date)
    generate_overview_table(la_overall, area_overall, location_name, radius_label, loc_out_dir,
                            raid_id=raid_id, raid_date=raid_date)

    # NAICS pie charts — data already collected above, no extra file reads
    create_naics_pie_chart(area_naics, location_name,
                           f'{loc_out_dir}/naics_pie_{location_name}.png')

    create_naics_pie_chart(la_naics, 'LA County',
                           f'{loc_out_dir}/naics_pie_LA.png')

    poi_csv = extract_poi_details(valid_files, area_gdf, location_name, raid_id=raid_id, out_dir=loc_out_dir, raid_date=raid_date)
    return area_overall, n_area_overall, twoweek_df, poi_csv


def combine_overview_tables(location_dirs, out_path):
    """Concatenate per-buffer analysis_overview_table_*.csv files into one combined CSV.
    The Raid_ID column (already written by generate_overview_table) is kept as the first column.
    """
    frames = []
    for loc_dir in location_dirs:
        pattern = os.path.join(loc_dir, 'analysis_overview_table_*.csv')
        for csv_path in glob.glob(pattern):
            try:
                df = pd.read_csv(csv_path)
                frames.append(df)
            except Exception as e:
                log_step(f"⚠️ Could not read {os.path.basename(csv_path)}: {str(e)[:80]}")
    if not frames:
        log_step("⚠️ No overview CSVs found to combine.")
        return None
    combined = pd.concat(frames, ignore_index=True)
    # Ensure Raid_ID is first column
    cols = ['Raid_ID'] + [c for c in combined.columns if c != 'Raid_ID']
    combined = combined[[c for c in cols if c in combined.columns]]
    combined.to_csv(out_path, index=False)
    log_step(f"📋 Combined overview → {os.path.basename(out_path)}  "
             f"({len(combined)} rows across {combined['Raid_ID'].nunique()} raid(s))")
    return combined


def run_all_buffers():
    half, quarter = get_buffer_polygons()
    log_step(f"🚀 Running analysis for {len(half)+len(quarter)} buffer polygons...")
    loc_dirs = []
    for radius, gdf in [('half_mile', half), ('quarter_mile', quarter)]:
        for idx, row in gdf.iterrows():
            raid_id       = str(row.get('ID', idx))
            location_name = f"{radius}_{row.get('Site', row.get('City',''))}_{raid_id}"
            loc_dir   = os.path.join(OUT_DIR, location_name.replace('/', '_').replace(' ', '_'))
            loc_cache = os.path.join(loc_dir, 'cache')
            loc_stats = os.path.join(loc_dir, 'descriptive_stats')
            os.makedirs(loc_dir,   exist_ok=True)
            os.makedirs(loc_cache, exist_ok=True)
            os.makedirs(loc_stats, exist_ok=True)
            area = gpd.GeoDataFrame([row], crs=gdf.crs)
            raid_date = parse_raid_date(row)
            run_single_area(area, location_name, loc_cache, loc_stats, loc_dir,
                            high_impact_include_ag=True, raid_id=raid_id, raid_date=raid_date)
            loc_dirs.append(loc_dir)
    combine_overview_tables(loc_dirs, os.path.join(OUT_DIR, 'combined_overview_all_buffers.csv'))
    log_step("🎉 ALL BUFFER LOCATIONS COMPLETE!")


# ============================================
# 🎮 BUFFER WIDGET
# ============================================

half, quarter = get_buffer_polygons()
buffer_options  = []
buffer_metadata = {}   # display_name -> (radius, idx, row)

for radius, gdf in [('half_mile', half), ('quarter_mile', quarter)]:
    for idx, row in gdf.iterrows():
        name = f"{radius} {row.get('Site','')} {row.get('City','')}_{row.get('ID',idx)}".strip()
        if not name:
            name = f"{radius}_{idx}"
        buffer_options.append(name)
        buffer_metadata[name] = (radius, idx, row)

location_multiselector = widgets.SelectMultiple(
    options=buffer_options,
    description='Select Buffers:',
    layout=widgets.Layout(width='620px', height='260px'),
    rows=15,
)

ag_checkbox    = widgets.Checkbox(value=True,  description='Include Agriculture in High Impact')
cache_checkbox = widgets.Checkbox(value=False, description='Clear cache before analysis')

run_single_btn = widgets.Button(description='🎯 Selected Buffers', button_style='info',    icon='map')
run_all_btn    = widgets.Button(description='🔥 ALL Buffers',      button_style='success', icon='analytics')

output         = widgets.Output()
selection_info = widgets.HTML(value=f"<b>Total buffers: {len(buffer_options)}</b> | Selected: 0")


def _clear_cache(loc_cache=None):
    try:
        target = loc_cache if (loc_cache and os.path.exists(loc_cache)) else CACHE_DIR
        for f in glob.glob(os.path.join(target, '*.pkl')):
            os.remove(f)
            log_step(f"🗑️ Deleted {os.path.basename(f)}")
    except Exception as e:
        log_step(f"⚠️ Error clearing cache: {str(e)[:80]}")


def update_selection_info(change=None):
    n = len(location_multiselector.value)
    selection_info.value = (f"<b>Total buffers: {len(buffer_options)}</b> | "
                            f"<b style='color:blue'>Selected: {n}</b>")


def run_selected(b):
    with output:
        clear_output()
        selected = location_multiselector.value
        if not selected:
            print("⚠️ No buffers selected.")
            return

        log_step(f"🚀 Running analysis for {len(selected)} selected buffer(s)...")
        loc_dirs = []
        poi_results = []  # (location_name, area_df) for bar chart
        twoweek_results = []
        poi_detail_paths = []

        for display_name in selected:
            radius, idx, row = buffer_metadata[display_name]
            gdf = half if radius == 'half_mile' else quarter

            raid_id       = str(row.get('ID', idx))
            location_name = f"{radius}_{row.get('Site', row.get('City',''))}_{row.get('ID',idx)}"
            loc_dir   = os.path.join(OUT_DIR, location_name.replace('/', '_').replace(' ', '_'))
            loc_cache = os.path.join(loc_dir, 'cache')
            loc_stats = os.path.join(loc_dir, 'descriptive_stats')
            os.makedirs(loc_dir,   exist_ok=True)
            os.makedirs(loc_cache, exist_ok=True)
            os.makedirs(loc_stats, exist_ok=True)

            if cache_checkbox.value:
                _clear_cache(loc_cache)
            area = gpd.GeoDataFrame([row], crs=gdf.crs)
            raid_date = parse_raid_date(row)
            result = run_single_area(area, location_name, loc_cache, loc_stats, loc_dir,
                                     ag_checkbox.value, raid_id=raid_id, raid_date=raid_date)
            if result:
                area_df, _, twoweek_df, poi_csv = result
            else:
                area_df, twoweek_df, poi_csv = None, None, None
            poi_results.append((location_name, area_df))
            if poi_csv:
                poi_detail_paths.append(poi_csv)
            if twoweek_df is not None:
                twoweek_results.append(twoweek_df)
            loc_dirs.append(loc_dir)

        # ── POI bar chart: 2024 vs 2025 ──
        if poi_results:
            bar_path = os.path.join(OUT_DIR, 'poi_bar_2024_vs_2025_selected.png')
            create_poi_bar_chart(poi_results, save_path=bar_path)

        # ── Combined overview table ──
        combined_path = os.path.join(OUT_DIR, 'combined_overview_selected.csv')
        combined_df   = combine_overview_tables(loc_dirs, combined_path)
        if combined_df is not None and not combined_df.empty:
            print(f"\n📋 Combined Overview Table "
                  f"({len(combined_df)} rows across {combined_df['Raid_ID'].nunique()} raid(s)):")
            display(combined_df)

        if twoweek_results:
            combined_2week = pd.concat(twoweek_results, ignore_index=True)

            # Keep only one LA County row at the end of the combined table
            la_df = combined_2week[combined_2week['Series'] == 'LA County']
            area_df = combined_2week[combined_2week['Series'] != 'LA County']
            if not la_df.empty:
                la_row = la_df.iloc[[0]].copy()
                combined_2week = pd.concat([area_df, la_row], ignore_index=True)

            combined_2week_path = os.path.join(OUT_DIR, 'combined_2week_prepost_selected.csv')
            combined_2week.to_csv(combined_2week_path, index=False)
            log_step(f"📊 Combined 2-week pre/post table saved: {os.path.basename(combined_2week_path)}")
            print(f"\n📋 Combined 2-week pre/post table ({len(combined_2week)} rows) saved and displayed:")
            display(combined_2week)

        if poi_detail_paths:
            all_poi = pd.concat([pd.read_csv(p) for p in poi_detail_paths], ignore_index=True)
            combined_poi_path = os.path.join(OUT_DIR, 'combined_poi_details_selected.csv')
            all_poi.to_csv(combined_poi_path, index=False)
            log_step(f"📋 Combined POI details saved: {os.path.basename(combined_poi_path)}")
            print(f"\n📋 Combined POI details ({len(all_poi)} rows) saved:")
            display(all_poi.head(20))
            try:
                geo_all = gpd.GeoDataFrame(
                    all_poi,
                    geometry=gpd.points_from_xy(all_poi['poi_lon'], all_poi['poi_lat']),
                    crs='EPSG:4326'
                )
                combined_poi_geo = os.path.join(OUT_DIR, 'combined_poi_details_selected.geojson')
                geo_all.to_file(combined_poi_geo, driver='GeoJSON')
                log_step(f"🗺️ Combined POI GeoJSON saved: {os.path.basename(combined_poi_geo)}")
            except Exception as e:
                log_step(f"⚠️ Could not write combined POI GeoJSON: {str(e)[:80]}")

        log_step(f"✅ Completed {len(selected)} buffer(s)!")

def run_all(b):
    with output:
        clear_output()
        if cache_checkbox.value:
            _clear_cache()
        run_all_buffers()


location_multiselector.observe(update_selection_info, names='value')
run_single_btn.on_click(run_selected)
run_all_btn.on_click(run_all)
update_selection_info()

display(widgets.VBox([
    widgets.HTML("<h2>🗺️ ICP Raid Buffer Analysis</h2>"),
    selection_info,
    widgets.HTML(f"<p>📂 Outputs → <code>{OUT_DIR}</code></p>"),
    location_multiselector,
    ag_checkbox,
    cache_checkbox,
    widgets.HBox([run_single_btn, run_all_btn]),
    output
]))


[19:08:30] 🔍 Loaded 11 half-mile buffers and 11 quarter-mile buffers


In [33]:
# ============================================
# 🔍 TEST: Inspect Advan Weekly CSV Column Names
# ============================================
# This cell reads the FIRST weekly foot-traffic CSV and prints ALL column names
# so we can identify the correct POI ID and name fields.

import glob, gzip, csv, os

files = sorted(glob.glob(os.path.join(WEEKLY_DATA_FOLDER, '*.csv*')))
print(f"📂 WEEKLY_DATA_FOLDER: {WEEKLY_DATA_FOLDER}")
print(f"   Found {len(files)} CSV file(s)\n")

if not files:
    print("⚠️ No files found — check WEEKLY_DATA_FOLDER path")
else:
    # Inspect first 2 files to confirm schema consistency
    for fp in files[:2]:
        print(f"━━━ {os.path.basename(fp)} ━━━")
        try:
            opener = gzip.open if fp.endswith('.gz') else open
            with opener(fp, 'rt', encoding='utf-8', errors='replace') as f:
                reader = csv.reader(f)
                headers = next(reader, None)
                if not headers:
                    print("  (empty file)\n")
                    continue

                print(f"  Total columns: {len(headers)}")
                print(f"  ALL COLUMNS:\n    {headers}\n")

                # Flag likely ID / name / zip columns
                id_kw = ['key', 'id', 'placekey', 'safegraph', 'identifier']
                name_kw = ['name', 'brand', 'location_name', 'business', 'store', 'place_name']
                zip_kw = ['zip', 'postal']

                print("  🔑 Likely POI-ID columns:")
                for h in headers:
                    if any(k in h.lower() for k in id_kw):
                        print(f"      → {h}")

                print("  🏷️  Likely POI-NAME columns:")
                for h in headers:
                    if any(k in h.lower() for k in name_kw):
                        print(f"      → {h}")

                print("  📮 Likely ZIPCODE columns:")
                for h in headers:
                    if any(k in h.lower() for k in zip_kw):
                        print(f"      → {h}")

                # Print first 3 data rows (truncated per field)
                print("\n  Sample rows (first 3):")
                for i, row in enumerate(reader):
                    if i >= 3:
                        break
                    # Show each field value alongside its header
                    print(f"  Row {i}:")
                    for h, v in zip(headers, row):
                        display_v = v[:80] + '…' if len(v) > 80 else v
                        print(f"    {h:>30s} = {display_v}")
                print()
        except Exception as e:
            print(f"  ⚠️ Error reading file: {e}\n")

print("✅ Use the column names above to update extract_poi_details().")
print("   The 'cols' list in that function must INCLUDE the ID/name columns")
print("   for them to appear in the output CSV.")


📂 WEEKLY_DATA_FOLDER: /primary/work/ice/temp/california-foot-traffic-weekly-patterns-plus/california-weekly-data-all/
   Found 362 CSV file(s)

━━━ 2019-01-07--data_01c22eee-0307-ac7d-0042-fa0705d7e07e_304_5_0.csv.gz ━━━
  Total columns: 38
  ALL COLUMNS:
    ['BRAND', 'BUCKETED_DWELL_TIMES', 'CITY', 'CLOSE_DATE', 'DATE_RANGE_END', 'DATE_RANGE_START', 'DEVICE_TYPE', 'DISTANCE_FROM_HOME', 'FOOTPRINT_ID', 'ID_STORE', 'ISO_COUNTRY_CODE', 'IS_DISTRIBUTOR', 'LATITUDE', 'LOCATION_NAME', 'LONGITUDE', 'MEDIAN_DWELL', 'MSA_CODE', 'NAICS_CODE', 'OPEN_DATE', 'PERSISTENT_ID', 'PERSISTENT_ID_STORE', 'POI_CBG', 'POSTAL_CODE', 'REGION', 'RELATED_SAME_DAY_BRAND', 'RELATED_SAME_WEEK_BRAND', 'STREET_ADDRESS', 'SUB_CATEGORY', 'TICKER', 'TOP_CATEGORY', 'VISITOR_COUNTRY_OF_ORIGIN', 'VISITOR_COUNTS', 'VISITOR_DAYTIME_CBGS', 'VISITOR_HOME_AGGREGATION', 'VISITOR_HOME_CBGS', 'VISITS_BY_DAY', 'VISITS_BY_EACH_HOUR', 'VISIT_COUNTS']

  🔑 Likely POI-ID columns:
      → FOOTPRINT_ID
      → ID_STORE
      → PERSIST